# Algoritmo de búsqueda de Grover

**Problema:** encontrar un elemento marcado dentro de una lista *desordenada* de $N = 2^n$ elementos.

| | Consultas necesarias |
|---|---|
| Clásico (fuerza bruta) | $O(N)$ — en promedio $N/2$ |
| Grover (cuántico) | $O(\\sqrt{N})$ |

**Idea:** se parte de una superposición uniforme (todos los elementos a la vez) y se repite un bloque de dos pasos:

1. **Oráculo** — reconoce el elemento buscado y le invierte el signo de su amplitud.
2. **Difusor** — refleja todas las amplitudes respecto al promedio, lo que *amplifica* la del elemento marcado.

Cada repetición gira el estado un poco más hacia la solución. El número óptimo de repeticiones es $\\lfloor \\frac{\\pi}{4}\\sqrt{N} \\rfloor$.

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc -q

In [ ]:
from math import floor, pi, sqrt
from qiskit import QuantumCircuit, transpile

N = 3             # 3 qubits  ->  2^3 = 8 elementos
OBJETIVO = "101"  # el elemento que buscamos (el 5 en decimal)
SHOTS = 1024

ITERACIONES = floor(pi / 4 * sqrt(2 ** N))
print(f"Espacio de búsqueda: {2**N} elementos | Objetivo: |{OBJETIVO}> | Iteraciones óptimas: {ITERACIONES}")

## 1. El oráculo

Marca el estado buscado cambiándole el signo: $|x\\rangle \\rightarrow -|x\\rangle$ sólo si $x$ es el objetivo.

Truco: una puerta `Z` multicontrolada sólo marca $|111\\rangle$. Para marcar cualquier otro estado, se ponen `X` en las posiciones donde el objetivo tiene un `0` (lo convierte temporalmente en $|111\\rangle$) y se deshacen después.

In [ ]:
def oraculo(objetivo):
    qc = QuantumCircuit(N, name="Oraculo")
    # Qiskit numera los qubits al revés que la cadena de bits -> reversed()
    ceros = [i for i, bit in enumerate(reversed(objetivo)) if bit == "0"]
    qc.x(ceros)
    qc.h(N - 1)
    qc.mcx(list(range(N - 1)), N - 1)   # H-MCX-H = Z multicontrolada
    qc.h(N - 1)
    qc.x(ceros)
    return qc

oraculo(OBJETIVO).draw("mpl")

## 2. El difusor (inversión sobre el promedio)

Refleja cada amplitud alrededor del promedio. Como la del estado marcado quedó negativa, el promedio baja un poco y esa amplitud salta muy por encima de las demás.

In [ ]:
def difusor():
    qc = QuantumCircuit(N, name="Difusor")
    qc.h(range(N))
    qc.x(range(N))
    qc.h(N - 1)
    qc.mcx(list(range(N - 1)), N - 1)
    qc.h(N - 1)
    qc.x(range(N))
    qc.h(range(N))
    return qc

difusor().draw("mpl")

## 3. El circuito completo

`H` en todos los qubits (superposición uniforme) + `ITERACIONES` veces (oráculo + difusor) + medición.

In [ ]:
qc = QuantumCircuit(N, N)
qc.h(range(N))                         # superposición uniforme
for _ in range(ITERACIONES):
    qc.compose(oraculo(OBJETIVO), inplace=True)
    qc.compose(difusor(), inplace=True)
qc.measure(range(N), range(N))

qc.draw("mpl", fold=-1)

## 4. Ejecución en el simulador ideal

Sin ruido: es la referencia teórica. Se espera ~94.5% de acierto con 2 iteraciones.

In [ ]:
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

def exito(counts):
    return 100 * counts.get(OBJETIVO, 0) / sum(counts.values())

sim = AerSimulator()
counts_ideal = sim.run(transpile(qc, sim), shots=SHOTS).result().get_counts()

print(f"Acierto: {exito(counts_ideal):.1f}%")
plot_histogram(counts_ideal, title="Simulador ideal")

## 5. Ejecución con el ruido de un chip real

Mismo circuito, pero usando el modelo de ruido calibrado de `ibm_sherbrooke`. Sirve para anticipar lo que hará el hardware sin gastar cuota de ejecución.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

sim_ruido = AerSimulator.from_backend(FakeSherbrooke())
counts_ruido = sim_ruido.run(transpile(qc, sim_ruido, optimization_level=3), shots=SHOTS).result().get_counts()

print(f"Acierto: {exito(counts_ruido):.1f}%")
plot_histogram(counts_ruido, title="Simulador con ruido de ibm_sherbrooke")

## 6. Ejecución en un computador cuántico real de IBM

Requiere una cuenta gratuita en [quantum.ibm.com](https://quantum.ibm.com). El token se guarda una sola vez:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="TU_TOKEN", overwrite=True)
```

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()                        # usa la cuenta guardada
backend = service.least_busy(operational=True, simulator=False)
print(f"Backend: {backend.name} | {backend.num_qubits} qubits | {backend.processor_type}")

qc_hw = transpile(qc, backend, optimization_level=3)    # adapta a la topología y puertas reales
print(f"Profundidad: {qc.depth()} -> {qc_hw.depth()}")
print(f"Puertas:     {dict(qc.count_ops())}")
print(f"          -> {dict(qc_hw.count_ops())}")

job = SamplerV2(mode=backend).run([qc_hw], shots=SHOTS)
print("\nJob ID:", job.job_id())
counts_real = job.result()[0].data.c.get_counts()

print(f"Acierto: {exito(counts_real):.1f}%")
plot_histogram(counts_real, title=f"Hardware real: {backend.name}")

## 7. Comparación y análisis

In [ ]:
plot_histogram([counts_ideal, counts_ruido, counts_real],
               legend=["Ideal (93.8%)", "Ruido simulado (73.5%)", "ibm_marrakesh real (61.2%)"],
               title="Grover: |101> domina en los tres casos",
               figsize=(13, 4), bar_labels=False)

### Lectura de los resultados

| Ejecución | Acierto en $|101\\rangle$ |
|---|---|
| Simulador ideal (Aer) | **93.8 %** (teórico 94.5 %) |
| Simulador con ruido de `ibm_sherbrooke` (Eagle) | **73.5 %** |
| **Hardware real `ibm_marrakesh` (Heron r2)** | **61.2 %** |
| Azar / clásico sin información | 12.5 % |

**El algoritmo funciona en hardware real.** $|101\\rangle$ sale 627 de 1024 veces: 6.5× más que el segundo
estado más frecuente y casi 5× por encima del 12.5 % del azar. Y todo con **2 consultas al oráculo**
frente a las ~4 de una búsqueda clásica promedio — el speedup $O(\\sqrt{N})$ vs $O(N)$.

**Por qué el ideal no llega al 100 %.** Grover es probabilístico: cada iteración rota el estado un ángulo
fijo hacia la solución, y con $N=8$ dos iteraciones lo dejan *casi* —pero no exactamente— sobre el eje
del objetivo. 94.5 % es el máximo teórico. Hacer **más** iteraciones **empeora** el resultado: el estado
se pasa de largo y empieza a alejarse (*sobre-rotación*).

**Por qué el hardware pierde 33 puntos.** La causa se ve en la transpilación:

| | Profundidad | Puertas | Composición |
|---|---|---|---|
| Circuito lógico | 22 | 46 | `h`×23, `x`×16, `ccx`×4 |
| Transpilado a `ibm_marrakesh` | **139** | **187** | `sx`×74, `rz`×62, **`cz`×39**, `x`×9 |

1. **Descomposición al conjunto nativo.** Heron sólo ejecuta `rz`, `sx`, `x` y `cz`. Las 4 Toffoli (`ccx`)
   del oráculo y el difusor no existen físicamente: cada una se descompone en ~6 `cz` más rotaciones.
   La profundidad se multiplica por 6.
2. **Error por puerta de dos qubits.** Esas 39 `cz` son el cuello de botella: con ~3×10⁻³ de error cada
   una se acumula ya un ~10 % sólo por ellas, y se compone multiplicativamente con el resto.
3. **Decoherencia y error de lectura.** Profundidad 139 da tiempo a que $T_1$/$T_2$ degraden el estado,
   y la medición añade su propio 1–2 %.

El efecto neto es que **la distribución se aplana**: el pico baja de 94 % a 61 % y los otros siete estados
suben desde casi cero hasta 4–9 % cada uno. Fíjate en que $|100\\rangle$ (9.4 %) destaca sobre el resto:
difiere del objetivo en un solo bit, la firma típica de un error de un qubit al final del circuito.

**Conclusión.** Grover demuestra la ventaja cuadrática de forma tangible, pero también deja ver el límite
de la era NISQ: con 3 qubits y 2 iteraciones el ruido ya cuesta un tercio del acierto. Escalar a espacios
de búsqueda grandes —donde la ventaja $\\sqrt{N}$ sería realmente útil— exige muchas más iteraciones y
circuitos más profundos de los que el hardware tolera sin corrección de errores. Ahí entran las
generaciones nuevas de IBM: **Heron** (el chip que usamos, con `cz` sintonizables y mucho menos
*crosstalk* que Eagle) y **Nighthawk**, cuya mayor conectividad reduce los `SWAP` que el transpilador
debe insertar y con ellos la profundidad efectiva. Menos profundidad y menos error por `cz` es
exactamente lo que Grover necesita para escalar.